In [1]:
import os
import time

import yaml

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR

from RLAlg.utils import set_seed_everywhere

from config import METAWORLD_CFGS, DMC_CFGS
from model.encoder import FrameObservationEncoderNet, EncoderNet
from state_frame_dataset import get_dataloader

In [2]:
class Alignment(nn.Module):
    def __init__(self, vector_weight, config):
        super().__init__()

        self.state_encoder = EncoderNet(15, config["encoder_layers"])

        self.frame_encoder = FrameObservationEncoderNet(6, self.state_encoder.dim)


        self.state_encoder.load_state_dict(vector_weight)

        self.state_encoder.eval()

        for param in self.state_encoder.parameters():
            param.requires_grad = False

    @torch.no_grad()
    def encode_states(self, vectors):
        state_features = self.state_encoder.get_features(vectors)

        return state_features[-1]

    def encode_frames(self, frames):
        frame_features = self.frame_encoder(frames, False, True)

        return frame_features
    
    def forward(self, frames, states):
        frame_features = self.encode_frames(frames)
        state_features = self.encode_states(states)

        return frame_features, state_features

In [3]:
task_name = "hopper"
seed = 0

In [4]:
if task_name in METAWORLD_CFGS:
    config_path = "configs/ddpg_metaworld.yaml"
elif task_name in DMC_CFGS:
    config_path = "configs/ddpg_dmc.yaml"

with open(config_path, "r") as file:
    config = yaml.safe_load(file)

weight_folder = f"weights/ddpg/{task_name}"
        
if not os.path.exists(weight_folder):
    os.makedirs(weight_folder)

set_seed_everywhere(0)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

encoder_weight, _, _ = torch.load(f"{weight_folder}/actor_best_{seed}.pt", weights_only=True)
model = Alignment(encoder_weight, config).to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = CosineAnnealingLR(optimizer=optimizer, T_max=20)
dataloader = get_dataloader(task_name)
size = len(dataloader.dataset)

In [5]:
def cosine_loss(x, y):
    return 1. - F.cosine_similarity(x, y, dim=-1).mean()

def mse_loss(x, y):
    return F.mse_loss(x, y, reduction="mean")

def info_nce_loss(img_feat, state_feat, T):
    img_norm   = F.normalize(img_feat,   dim=-1)
    state_norm = F.normalize(state_feat, dim=-1)

    logits = (img_norm @ state_norm.t()) / T          
    labels = torch.arange(logits.size(0),
                          device=logits.device)
    return (F.cross_entropy(logits, labels) +
            F.cross_entropy(logits.t(), labels)) * 0.5

In [6]:
for _ in range(20):
    start_time = time.time()
    running_loss = 0.0
    for _, (vectors, frames) in enumerate(dataloader):
        vectors = vectors.to(device)
        frames = frames.to(device)
        
        frame_features, vector_features = model(frames, vectors)
        
        loss = 0.5 * mse_loss(frame_features, vector_features) + 0.5 * cosine_loss(frame_features, vector_features)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * vectors.size(0)
    
    scheduler.step()
    end_time = time.time()
    train_loss = running_loss / size
    print(f"Train Loss: {train_loss:.4f}")
    elapsed_time = end_time - start_time
    print(f"The loop took {elapsed_time:.2f} seconds to complete.")

torch.save(model.frame_encoder.state_dict(), f"weights/ddpg/pickplace/frame_encoder_{seed}_cos.pt")

Train Loss: 0.0903
The loop took 65.49 seconds to complete.
Train Loss: 0.0522
The loop took 48.24 seconds to complete.
Train Loss: 0.0490
The loop took 46.52 seconds to complete.
Train Loss: 0.0468
The loop took 45.66 seconds to complete.
Train Loss: 0.0450
The loop took 45.60 seconds to complete.
Train Loss: 0.0434
The loop took 45.66 seconds to complete.
Train Loss: 0.0419
The loop took 45.80 seconds to complete.
Train Loss: 0.0401
The loop took 45.68 seconds to complete.
Train Loss: 0.0384
The loop took 46.07 seconds to complete.
Train Loss: 0.0367
The loop took 45.81 seconds to complete.
Train Loss: 0.0348
The loop took 45.59 seconds to complete.
Train Loss: 0.0327
The loop took 45.57 seconds to complete.
Train Loss: 0.0306
The loop took 45.58 seconds to complete.
Train Loss: 0.0282
The loop took 46.11 seconds to complete.
Train Loss: 0.0260
The loop took 46.03 seconds to complete.
Train Loss: 0.0236
The loop took 45.70 seconds to complete.
Train Loss: 0.0213
The loop took 45.50 s